In [1]:
# ============================================================
# CELL 1 — Title and Setup
# ============================================================
# Chapter 15: Sensitivity Analysis — How Wrong Could We Be?
# Causal Inference with LLMs
# Author: Hrishi Pal
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy.stats import norm, ttest_ind
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Project-relative figure output (works when kernel cwd is this folder)
PROJECT_ROOT = Path.cwd()
FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 150
})

print("All libraries loaded successfully.")
print(f"Figures directory: {FIG_DIR.resolve()}")


All libraries loaded successfully.
Figures directory: /Users/hrishipal/Desktop/causal_inference_ch15/figures


In [ ]:
# ============================================================
# CELL 2 — AI SCAFFOLD WITH MANDATORY HUMAN DECISION NODE
# ============================================================
# LLM proposes variables and DAG edges; you validate before analysis.
# ============================================================

import json
import urllib.request

def call_claude(prompt):
    api_key = os.environ.get("ANTHROPIC_API_KEY", "")
    url = "https://api.anthropic.com/v1/messages"
    payload = json.dumps({
        "model": "claude-sonnet-4-20250514",
        "max_tokens": 1000,
        "messages": [{"role": "user", "content": prompt}]
    }).encode()
    req = urllib.request.Request(url, data=payload, headers={
        "Content-Type": "application/json",
        "anthropic-version": "2023-06-01",
        "x-api-key": api_key,
    })
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())["content"][0]["text"]

DATA_DICTIONARY = """
Variables in the hospital readmission dataset:
- age: patient age in years
- los: length of hospital stay (days)
- chronic: binary flag for chronic condition
- insured: binary flag for insurance status
- severity: continuous severity score (z-scored)
- treatment: received follow-up call (1) or not (0)
- outcome: no readmission within 30 days (1) or readmission (0)
"""

SCAFFOLD_PROMPT = f"""
You are a causal inference assistant. Given this data dictionary:

{DATA_DICTIONARY}

Do TWO things:
1. List which variables should be included as confounders in a
   propensity score model predicting treatment (follow-up call).
2. Propose DAG edges as a list of tuples (cause, effect) for the
   causal graph of this study.

Return ONLY a JSON object with keys "confounders" and "dag_edges".
No preamble, no explanation, no markdown.
"""

print("Calling LLM to enumerate variables and propose DAG edges...")
print("=" * 60)

try:
    llm_response = call_claude(SCAFFOLD_PROMPT)
    proposed = json.loads(llm_response)
    print("LLM proposed confounders:")
    for c in proposed["confounders"]:
        print(f"  - {c}")
    print("\nLLM proposed DAG edges:")
    for edge in proposed["dag_edges"]:
        print(f"  {edge[0]} → {edge[1]}")
except Exception as e:
    print("API not available — using pre-specified proposal for demonstration.")
    proposed = {
        "confounders": ["age", "los", "chronic", "insured", "severity"],
        "dag_edges": [
            ["insured", "treatment"],
            ["severity", "treatment"],
            ["severity", "outcome"],
            ["chronic", "outcome"],
            ["treatment", "outcome"],
            ["age", "outcome"],
            ["los", "treatment"]
        ]
    }
    for c in proposed["confounders"]:
        print(f"  - {c}")
    print("\nLLM proposed DAG edges:")
    for edge in proposed["dag_edges"]:
        print(f"  {edge[0]} → {edge[1]}")

print("=" * 60)
print()

# ============================================================
# MANDATORY HUMAN DECISION NODE
# ============================================================

VALIDATED_CONFOUNDERS = [
    "age",
    "los",
    "chronic",
    "insured",
    "severity"
]

VALIDATED_DAG_EDGES = [
    # ("insured", "treatment"),
    # ("severity", "treatment"),
    # ("severity", "outcome"),
    # ("chronic", "outcome"),
    # ("treatment", "outcome"),
    # ("age", "outcome"),
    # REJECTED: ("los", "treatment") — los does not cause call assignment
    # REJECTED: ("age", "treatment") — age not a protocol driver
]

print("Human Decision Node complete.")
print(f"Validated confounders: {VALIDATED_CONFOUNDERS}")
print("Proceeding with validated variable set.")
print()
print("NOTE: The LLM proposed 'los → treatment' as a DAG edge.")
print("This was REJECTED. Length of stay does not cause call")
print("assignment — staffing protocol does. This is an unmeasured")
print("confounder. los is retained as a confounder of outcome only.")


Calling LLM to enumerate variables and propose DAG edges...


API not available — using pre-specified proposal for demonstration.
  - age
  - los
  - chronic
  - insured
  - severity

LLM proposed DAG edges:
  insured → treatment
  severity → treatment
  severity → outcome
  chronic → outcome
  treatment → outcome
  age → outcome
  los → treatment

Human Decision Node complete.
Validated confounders: ['age', 'los', 'chronic', 'insured', 'severity']
Proceeding with validated variable set.

NOTE: The LLM proposed 'los → treatment' as a DAG edge.
This was REJECTED. Length of stay does not cause call
assignment — staffing protocol does. This is an unmeasured
confounder. los is retained as a confounder of outcome only.


In [3]:
# ============================================================
# CELL 3 — Generate Synthetic Dataset
# ============================================================

rng = np.random.default_rng(42)
n = 2000

age      = rng.normal(62, 12, n)
los      = rng.exponential(4, n)
chronic  = rng.binomial(1, 0.55, n)
insured  = rng.binomial(1, 0.78, n)
severity = rng.normal(0, 1, n)

log_odds_tx = -0.5 + 0.6 * insured - 0.4 * severity - 0.02 * los
p_treatment = 1 / (1 + np.exp(-log_odds_tx))
treatment   = rng.binomial(1, p_treatment, n)

log_odds_y = 0.8 + 0.35 * treatment - 0.4 * severity - 0.25 * chronic
p_outcome  = 1 / (1 + np.exp(-log_odds_y))
outcome    = rng.binomial(1, p_outcome, n)

df = pd.DataFrame({
    'age': age, 'los': los, 'chronic': chronic,
    'insured': insured, 'severity': severity,
    'treatment': treatment, 'outcome': outcome
})

print(f"Dataset generated: n={n}")
print(f"Treatment rate: {treatment.mean():.3f}")
print(f"Outcome rate (no readmission): {outcome.mean():.3f}")
print(df.describe().round(3))


Dataset generated: n=2000
Treatment rate: 0.483
Outcome rate (no readmission): 0.691
            age       los   chronic   insured  severity  treatment   outcome
count  2000.000  2000.000  2000.000  2000.000  2000.000   2000.000  2000.000
mean     61.338     3.916     0.568     0.784    -0.014      0.483     0.690
std      12.025     3.891     0.495     0.412     1.023      0.500     0.462
min      18.219     0.001     0.000     0.000    -3.529      0.000     0.000
25%      53.360     1.120     0.000     1.000    -0.682      0.000     0.000
50%      61.730     2.833     1.000     1.000    -0.017      0.000     1.000
75%      69.193     5.404     1.000     1.000     0.650      1.000     1.000
max     100.146    38.072     1.000     1.000     4.026      1.000     1.000


In [4]:
# ============================================================
# CELL 4 — Naive Logistic Regression (Unadjusted)
# ============================================================

X_naive = treatment.reshape(-1, 1)
naive_model = LogisticRegression(max_iter=500).fit(X_naive, outcome)
naive_or = np.exp(naive_model.coef_[0][0])

print("Naive logistic regression (unadjusted):")
print(f"  Odds ratio for treatment: {naive_or:.3f}")
print()

print("Adjusted logistic regression (treatment + confounders):")
X_full = df[VALIDATED_CONFOUNDERS + ['treatment']].values
full_model = LogisticRegression(max_iter=500).fit(X_full, outcome)
tx_idx = len(VALIDATED_CONFOUNDERS)
adj_or = np.exp(full_model.coef_[0][tx_idx])

n_boot = 1000
boot_ors = []
for _ in range(n_boot):
    idx = rng.integers(0, n, n)
    m = LogisticRegression(max_iter=500).fit(X_full[idx], outcome[idx])
    boot_ors.append(np.exp(m.coef_[0][tx_idx]))

ci_lo, ci_hi = np.percentile(boot_ors, [2.5, 97.5])
print(f"  Adjusted OR: {adj_or:.3f} (95% CI: {ci_lo:.3f}, {ci_hi:.3f})")
print("  Interpretation: follow-up call associated with")
if adj_or >= 1:
    print(f"  a {(adj_or - 1) * 100:.1f}% increase in odds of avoiding readmission (adjusted).")
else:
    print(f"  a {(1 - adj_or) * 100:.1f}% decrease in odds of avoiding readmission (adjusted).")


Naive logistic regression (unadjusted):
  Odds ratio for treatment: 1.675

Adjusted logistic regression (treatment + confounders):


  Adjusted OR: 1.463 (95% CI: 1.189, 1.778)
  Interpretation: follow-up call associated with
  a 46.3% increase in odds of avoiding readmission (adjusted).


In [ ]:
# ============================================================
# CELL 5 — AIPW / Doubly Robust Estimator
# ============================================================

def aipw_ate(df, outcome_col, treatment_col, covariate_cols):
    """
    Augmented Inverse Probability Weighting (AIPW) estimator.
    Returns Average Treatment Effect (ATE) with standard error.
    """
    n = len(df)
    X = df[covariate_cols].values
    A = df[treatment_col].values
    Y = df[outcome_col].values

    ps_model = LogisticRegression(max_iter=500).fit(X, A)
    e_x = ps_model.predict_proba(X)[:, 1]
    e_x = np.clip(e_x, 0.01, 0.99)

    X1 = np.column_stack([X, np.ones(n)])
    X0 = np.column_stack([X, np.zeros(n)])
    Xa = np.column_stack([X, A])

    out_model = LogisticRegression(max_iter=500).fit(Xa, Y)
    mu1 = out_model.predict_proba(X1)[:, 1]
    mu0 = out_model.predict_proba(X0)[:, 1]

    psi1 = mu1 + A * (Y - mu1) / e_x
    psi0 = mu0 + (1 - A) * (Y - mu0) / (1 - e_x)
    psi  = psi1 - psi0

    ate = psi.mean()
    se  = psi.std() / np.sqrt(n)
    return ate, se

ate, se = aipw_ate(df, 'outcome', 'treatment', VALIDATED_CONFOUNDERS)
z = 1.96
print("AIPW / Doubly Robust Estimator")
print("=" * 40)
print(f"  ATE (Average Treatment Effect): {ate:.4f}")
print(f"  Standard Error:                 {se:.4f}")
print(f"  95% CI: ({ate - z*se:.4f}, {ate + z*se:.4f})")
print()
print("Interpretation: receiving a follow-up call is associated")
print(f"with a {ate:.1%} increase in the probability of avoiding")
print("readmission, after doubly robust adjustment.")
print()
print("Note: This estimate still assumes no unmeasured confounding.")
print("The sensitivity analysis below addresses that assumption.")


AIPW / Doubly Robust Estimator
  ATE (Average Treatment Effect): 0.0781
  Standard Error:                 0.0208
  95% CI: (0.0372, 0.1189)

Interpretation: receiving a follow-up call is associated
with a 7.8% increase in the probability of avoiding
readmission, after doubly robust adjustment.

Note: This estimate still assumes no unmeasured confounding.
The sensitivity analysis below addresses that assumption.


In [6]:
# ============================================================
# CELL 6 — E-Value Computation
# ============================================================

def evalue(rr):
    """
    E-value for a risk ratio >= 1.
    For protective effects, pass 1/RR.
    """
    if rr < 1:
        raise ValueError("Pass 1/RR for protective effects.")
    return rr + np.sqrt(rr * (rr - 1))

or_estimate = 0.71
or_ci_upper = 0.87

rr_estimate = 1 / or_estimate
rr_ci_bound = 1 / or_ci_upper

ev_estimate = evalue(rr_estimate)
ev_ci_bound = evalue(rr_ci_bound)

print("E-Value Sensitivity Analysis")
print("=" * 40)
print(f"Risk ratio (point estimate):  {rr_estimate:.3f}")
print(f"Risk ratio (CI bound):        {rr_ci_bound:.3f}")
print(f"\nE-value (point estimate):     {ev_estimate:.2f}")
print(f"E-value (CI bound):           {ev_ci_bound:.2f}")
print()
print("Interpretation:")
print(f"  An unmeasured confounder would need RR >= {ev_estimate:.2f}")
print(f"  with BOTH treatment and outcome to explain away")
print(f"  the point estimate.")
print(f"  The CI bound requires only RR >= {ev_ci_bound:.2f} —")
print(f"  achievable by health literacy or medication adherence.")


E-Value Sensitivity Analysis
Risk ratio (point estimate):  1.408
Risk ratio (CI bound):        1.149

E-value (point estimate):     2.17
E-value (CI bound):           1.56

Interpretation:
  An unmeasured confounder would need RR >= 2.17
  with BOTH treatment and outcome to explain away
  the point estimate.
  The CI bound requires only RR >= 1.56 —
  achievable by health literacy or medication adherence.


In [7]:
# ============================================================
# CELL 7 — Rosenbaum Bounds
# ============================================================

def rosenbaum_pvalue(diffs, gamma):
    pos_diffs = diffs[diffs != 0]
    n = len(pos_diffs)
    if n == 0:
        return 1.0
    ranks   = np.arange(1, n + 1)
    p_upper = gamma / (1 + gamma)
    mu      = p_upper * n * (n + 1) / 2
    sigma   = np.sqrt(p_upper * (1 - p_upper) * n * (n + 1) * (2*n + 1) / 6)
    t_plus  = np.sum(ranks[pos_diffs > 0])
    z       = (t_plus - mu) / sigma
    return 1 - norm.cdf(z)

rng_rb = np.random.default_rng(42)
n_pairs = 500
treat_outcomes   = rng_rb.binomial(1, 0.81, n_pairs)
control_outcomes = rng_rb.binomial(1, 0.72, n_pairs)
diffs = treat_outcomes - control_outcomes

gammas  = np.arange(1.0, 3.05, 0.1)
pvalues = [rosenbaum_pvalue(diffs, g) for g in gammas]
results = pd.DataFrame({"Gamma": gammas, "Worst_case_p": pvalues})
sig = results[results["Worst_case_p"] >= 0.05]
threshold = float(sig["Gamma"].min()) if len(sig) else float("nan")

target = [1.0, 1.3, 1.5, 1.8, 2.0, 2.5, 3.0]
mask = results["Gamma"].apply(lambda x: any(abs(x - t) < 0.001 for t in target))
print("Rosenbaum Bounds Analysis")
print("=" * 40)
print(results[mask].to_string(index=False))
print(f"\nRosenbaum bound: significance lost at Γ = {threshold:.1f}")
print()
print("Interpretation:")
print(f"  A hidden factor shifting treatment odds by only")
print(f"  {(threshold-1)*100:.0f}% between matched pairs could undo the finding.")


Rosenbaum Bounds Analysis
 Gamma  Worst_case_p
   1.0      0.000027
   1.3      0.006288
   1.5      0.047245
   1.8      0.262729
   2.0      0.484637
   2.5      0.889968
   3.0      0.988410

Rosenbaum bound: significance lost at Γ = 1.6

Interpretation:
  A hidden factor shifting treatment odds by only
  60% between matched pairs could undo the finding.


In [8]:
# ============================================================
# CELL 8 — Positivity / Overlap Diagnostic
# ============================================================

X_ps = df[VALIDATED_CONFOUNDERS].values
ps_model = LogisticRegression(max_iter=500).fit(X_ps, treatment)
pscore = ps_model.predict_proba(X_ps)[:, 1]

df['pscore'] = pscore
treated_ps = df[df.treatment == 1]['pscore']
control_ps = df[df.treatment == 0]['pscore']

common_min = max(treated_ps.min(), control_ps.min())
common_max = min(treated_ps.max(), control_ps.max())
outside    = df[(df.pscore < common_min) | (df.pscore > common_max)]

print("Positivity / Overlap Diagnostic")
print("=" * 40)
print(pd.DataFrame({
    'Treated': treated_ps.describe(),
    'Control': control_ps.describe()
}).round(3))
print(f"\nUnits outside common support: {len(outside)} ({100*len(outside)/n:.1f}%)")
print(f"Common support region: [{common_min:.3f}, {common_max:.3f}]")
print()
if len(outside) > 0:
    print(f"ACTION: Trim {len(outside)} units before outcome analysis.")
    print("These units have no comparable counterpart and contaminate")
    print("the sensitivity statistics if included.")
else:
    print("Common support satisfied — no trimming required.")

df_trim = df[(df.pscore >= common_min) & (df.pscore <= common_max)].copy()
print(f"\nTrimmed dataset: n={len(df_trim)}")


Positivity / Overlap Diagnostic
       Treated   Control
count  966.000  1034.000
mean     0.508     0.460
std      0.107     0.107
min      0.130     0.180
25%      0.440     0.388
50%      0.511     0.460
75%      0.583     0.536
max      0.812     0.762

Units outside common support: 4 (0.2%)
Common support region: [0.180, 0.762]

ACTION: Trim 4 units before outcome analysis.
These units have no comparable counterpart and contaminate
the sensitivity statistics if included.

Trimmed dataset: n=1996


In [9]:
# ============================================================
# CELL 9 — Falsification Test
# ============================================================

rng3 = np.random.default_rng(99)
complication_binary = rng3.binomial(1, 0.12, n)

treated_comp = complication_binary[treatment == 1]
control_comp = complication_binary[treatment == 0]
t_stat, p_val = ttest_ind(treated_comp, control_comp)

print("Falsification Test: In-Hospital Complication Rate")
print("=" * 40)
print(f"  Treated mean:  {treated_comp.mean():.4f}")
print(f"  Control mean:  {control_comp.mean():.4f}")
print(f"  Difference:    {treated_comp.mean() - control_comp.mean():.4f}")
print(f"  p-value:       {p_val:.4f}")
print()
if p_val < 0.05:
    print("WARNING: Significant placebo effect detected.")
    print("Treatment variable may be proxying an unmeasured confounder.")
    print("Do not proceed to causal interpretation without investigation.")
else:
    print("Placebo test passed: no significant pre-discharge effect.")
    print("Causal story is consistent with this falsification check.")


Falsification Test: In-Hospital Complication Rate
  Treated mean:  0.1356
  Control mean:  0.1267
  Difference:    0.0089
  p-value:       0.5549

Placebo test passed: no significant pre-discharge effect.
Causal story is consistent with this falsification check.


In [10]:
# ============================================================
# CELL 10 — Figure 1: E-Value Joint Threshold Space
# ============================================================

def evalue_plot(rr):
    return rr + np.sqrt(rr * (rr - 1))

rr_vals = np.linspace(1.001, 4.0, 300)
curve_y = evalue_plot(rr_vals)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(rr_vals, curve_y, color='#378ADD', linewidth=2, label='E-value boundary')
ax.fill_between(rr_vals, curve_y, 4.0, alpha=0.10, color='#E24B4A', label='Can explain effect')
ax.fill_between(rr_vals, 1.0, curve_y, alpha=0.08, color='#378ADD', label='Cannot explain effect')
ax.plot(2.17, 2.17, 'o', color='#378ADD', markersize=9, zorder=5, label='E-value (2.17, 2.17)')
ax.annotate('E-value\n(2.17, 2.17)', xy=(2.17, 2.17), xytext=(2.5, 1.6),
            fontsize=9, color='#378ADD',
            arrowprops=dict(arrowstyle='->', color='#378ADD', lw=1.2))
ax.plot(2.0, 2.0, '^', color='#EF9F27', markersize=9, zorder=5, label='Medication adherence (~2.0)')
ax.annotate('Med. adherence\n(~2.0, 2.0)', xy=(2.0, 2.0), xytext=(2.4, 2.5),
            fontsize=9, color='#EF9F27',
            arrowprops=dict(arrowstyle='->', color='#EF9F27', lw=1.2))
ax.set_xlim(1, 4); ax.set_ylim(1, 4)
ax.set_xlabel('Confounder RR with treatment', fontsize=11)
ax.set_ylabel('Confounder RR with outcome', fontsize=11)
ax.set_title('Figure 1 — E-value joint threshold space', fontsize=12, fontweight='normal')
ax.legend(fontsize=8, loc='upper left')
ax.text(3.2, 1.3, 'Cannot\nexplain', fontsize=9, color='#378ADD', alpha=0.7)
ax.text(1.5, 3.5, 'Can\nexplain', fontsize=9, color='#E24B4A', alpha=0.7)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_evalue_threshold.png', bbox_inches='tight')
plt.show()


In [11]:
# ============================================================
# CELL 11 — Figure 2: Rosenbaum Γ Decay Curve
# ============================================================

gammas_plot = [1.0, 1.3, 1.5, 1.8, 2.0, 2.5, 3.0]
pvals_plot  = [0.000027, 0.006288, 0.047245, 0.262729, 0.484637, 0.889968, 0.988410]

fig, ax = plt.subplots(figsize=(7, 4))
colors_rb = ['#1D9E75' if p < 0.05 else '#E24B4A' for p in pvals_plot]
ax.plot(gammas_plot, pvals_plot, color='#1D9E75', linewidth=2.5, zorder=3)
ax.scatter(gammas_plot, pvals_plot, color=colors_rb, zorder=4, s=60)
ax.axhline(0.05, color='#E24B4A', linestyle='--', linewidth=1.5, label='α = 0.05')
ax.axvline(1.6, color='#888780', linestyle=':', linewidth=1.2, label='Γ = 1.6 (bound)')
ax.fill_between(gammas_plot, pvals_plot, 0.05,
                where=[p < 0.05 for p in pvals_plot],
                alpha=0.12, color='#1D9E75', label='Significant region')
ax.annotate('Significance lost\nΓ = 1.6', xy=(1.6, 0.05), xytext=(2.0, 0.12),
            fontsize=9, color='#888780',
            arrowprops=dict(arrowstyle='->', color='#888780', lw=1.2))
ax.set_xlim(1.0, 3.0); ax.set_ylim(0, 0.55)
ax.set_xlabel('Γ (hidden bias parameter)', fontsize=11)
ax.set_ylabel('Worst-case p-value', fontsize=11)
ax.set_title('Figure 2 — Rosenbaum Γ decay curve', fontsize=12, fontweight='normal')
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_rosenbaum_curve.png', bbox_inches='tight')
plt.show()


In [12]:
# ============================================================
# CELL 12 — Figure 3: E-Value Benchmark Bar Chart
# ============================================================

labels = ['E-value: point estimate', 'E-value: CI bound',
          'Medication adherence\n(benchmark)', 'Insurance status\n(measured)']
values = [2.17, 1.56, 2.0, 1.6]
colors_bar = ['#378ADD', '#378ADD', '#EF9F27', '#EF9F27']

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(labels, values, color=colors_bar, height=0.5, left=1.0)
ax.set_xlim(1.0, 2.6)
ax.axvline(1.0, color='#888780', linewidth=1)
for bar, val in zip(bars, values):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=10, color='#444441')
ax.set_xlabel('Risk ratio (null = 1.0)', fontsize=11)
ax.set_title('Figure 3 — E-value thresholds vs. benchmark confounders',
             fontsize=12, fontweight='normal')
blue_patch  = mpatches.Patch(color='#378ADD', label='E-value thresholds')
amber_patch = mpatches.Patch(color='#EF9F27', label='Benchmark confounders')
ax.legend(handles=[blue_patch, amber_patch], fontsize=9, loc='lower right')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_evalue_benchmarks.png', bbox_inches='tight')
plt.show()


In [13]:
# ============================================================
# CELL 13 — Figure 4: Propensity Score Common Support
# ============================================================

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(0, 1, 40)
ax.hist(treated_ps, bins=bins, alpha=0.55, color='#378ADD', label='Treated', density=True)
ax.hist(control_ps, bins=bins, alpha=0.45, color='#888780', label='Control', density=True)
ax.axvline(common_min, color='#E24B4A', linestyle='--', linewidth=1.5)
ax.axvline(common_max, color='#E24B4A', linestyle='--', linewidth=1.5,
           label=f'Common support [{common_min:.2f}, {common_max:.2f}]')
ymax = ax.get_ylim()[1]
ax.fill_betweenx([0, ymax], common_min, common_max, alpha=0.06, color='#1D9E75')
ax.set_xlabel('Estimated propensity score P(treatment | covariates)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Figure 4 — Propensity score common support diagnostic',
             fontsize=12, fontweight='normal')
ax.legend(fontsize=9)
ax.text(0.05, 0.85, 'Outside\nsupport', transform=ax.transAxes,
        fontsize=8, color='#E24B4A', alpha=0.8)
ax.text(0.82, 0.85, 'Outside\nsupport', transform=ax.transAxes,
        fontsize=8, color='#E24B4A', alpha=0.8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_common_support.png', bbox_inches='tight')
plt.show()

print("All figures saved to figures/")


All figures saved to figures/


## Author's note — Exercise 15.1 (known limitation)

The sweep below is meant to stress positivity, trimming, and matching. **Under the current data-generating process and the Rosenbaum routine implemented here, the reported Γ bound typically *increases* as weekend contamination rises (e.g. 1.7 → 2.0), instead of *decreasing* as the chapter's "What You Should Find" section suggests when overlap is violated.** This notebook therefore does **not** reproduce that directional story without further changes to the DGP or the sensitivity statistic. **Treat the mismatch as a known limitation:** use the exercise for mechanics and discussion, or revise the simulation so Γ and E-values degrade in the direction the chapter describes.

**Guided question Q1** assumes bounds *fall* with contamination; with the present code they may not—answer Q1 using the numbers *you* get and contrast them with the chapter's expected narrative.


In [14]:
# ============================================================
# CELL 14 — EXERCISE 15.1: Break the Model
# ============================================================
# Students: replace the three solution blocks with your own code,
# or study the reference implementation below.
# ============================================================

rng_ex = np.random.default_rng(42)

def evalue_ex(rr):
    if rr < 1:
        raise ValueError("Pass 1/RR for protective effects.")
    return rr + np.sqrt(rr * (rr - 1))

def rosenbaum_pvalue_ex(diffs, gamma):
    pos_diffs = diffs[diffs != 0]
    n = len(pos_diffs)
    if n == 0:
        return 1.0
    ranks   = np.arange(1, n + 1)
    p_upper = gamma / (1 + gamma)
    mu      = p_upper * n * (n + 1) / 2
    sigma   = np.sqrt(p_upper * (1 - p_upper) * n * (n + 1) * (2*n + 1) / 6)
    t_plus  = np.sum(ranks[pos_diffs > 0])
    z       = (t_plus - mu) / sigma
    return 1 - norm.cdf(z)

def rosenbaum_bound_ex(diffs, gamma_max=4.0, step=0.1):
    gammas = np.arange(1.0, gamma_max + step, step)
    for g in gammas:
        if rosenbaum_pvalue_ex(diffs, g) >= 0.05:
            return round(g, 1)
    return f"> {gamma_max}"

def nn_match_pair_diffs(df_trim):
    """1:1 nearest-neighbor on pscore, no replacement; return treated - control outcome."""
    treated = df_trim[df_trim.treatment == 1].reset_index(drop=True)
    control = df_trim[df_trim.treatment == 0].reset_index(drop=True)
    if len(treated) == 0 or len(control) == 0:
        return np.array([])
    c_ps = control['pscore'].to_numpy()
    c_out = control['outcome'].to_numpy()
    used = np.zeros(len(control), dtype=bool)
    diffs_list = []
    order = np.argsort(treated['pscore'].to_numpy())
    for i in order:
        ps = float(treated.loc[i, 'pscore'])
        dist = np.abs(c_ps - ps)
        dist[used] = np.inf
        j = int(np.argmin(dist))
        if not np.isfinite(dist[j]):
            continue
        used[j] = True
        diffs_list.append(float(treated.loc[i, 'outcome'] - c_out[j]))
    return np.array(diffs_list, dtype=float)

# Clean baseline
n_clean = 1800
age_c      = rng_ex.normal(63, 11, n_clean)
los_c      = rng_ex.exponential(4, n_clean)
chronic_c  = rng_ex.binomial(1, 0.55, n_clean)
insured_c  = rng_ex.binomial(1, 0.82, n_clean)
severity_c = rng_ex.normal(0, 1, n_clean)
log_odds_c = -0.4 + 0.6*insured_c - 0.4*severity_c - 0.02*los_c
p_tx_c     = 1 / (1 + np.exp(-log_odds_c))
treatment_c = rng_ex.binomial(1, p_tx_c, n_clean)
log_odds_yc = 1.2 + 0.4*treatment_c - 0.3*severity_c - 0.2*chronic_c
p_out_c     = 1 / (1 + np.exp(-log_odds_yc))
outcome_c   = rng_ex.binomial(1, p_out_c, n_clean)

df_clean = pd.DataFrame({
    'age': age_c, 'los': los_c, 'chronic': chronic_c,
    'insured': insured_c, 'severity': severity_c,
    'treatment': treatment_c, 'outcome': outcome_c,
    'weekend': np.zeros(n_clean, dtype=int)
})

n_wk = 400
age_w      = rng_ex.normal(68, 14, n_wk)
los_w      = rng_ex.exponential(5, n_wk)
chronic_w  = rng_ex.binomial(1, 0.70, n_wk)
insured_w  = rng_ex.binomial(1, 0.60, n_wk)
severity_w = rng_ex.normal(0.5, 1.2, n_wk)
log_odds_w  = -2.5 + 0.4*insured_w - 0.5*severity_w
p_tx_w      = 1 / (1 + np.exp(-log_odds_w))
treatment_w = rng_ex.binomial(1, p_tx_w, n_wk)
log_odds_yw = 0.6 + 0.4*treatment_w - 0.5*severity_w - 0.4*chronic_w
p_out_w     = 1 / (1 + np.exp(-log_odds_yw))
outcome_w   = rng_ex.binomial(1, p_out_w, n_wk)

df_weekend = pd.DataFrame({
    'age': age_w, 'los': los_w, 'chronic': chronic_w,
    'insured': insured_w, 'severity': severity_w,
    'treatment': treatment_w, 'outcome': outcome_w,
    'weekend': np.ones(n_wk, dtype=int)
})

def run_analysis(contamination_fraction):
    n_add     = int(contamination_fraction * len(df_weekend))
    df_contam = pd.concat([df_clean, df_weekend.iloc[:n_add]], ignore_index=True)
    X = df_contam[['age','los','chronic','insured','severity']].values
    y = df_contam['treatment'].values

    # (1) Propensity score
    ps_mod = LogisticRegression(max_iter=500).fit(X, y)
    pscore = ps_mod.predict_proba(X)[:, 1]

    df_contam = df_contam.copy()
    df_contam['pscore'] = pscore

    min_ps = max(df_contam[df_contam.treatment==1]['pscore'].min(),
                 df_contam[df_contam.treatment==0]['pscore'].min())
    max_ps = min(df_contam[df_contam.treatment==1]['pscore'].max(),
                 df_contam[df_contam.treatment==0]['pscore'].max())
    df_trim = df_contam[(df_contam.pscore >= min_ps) &
                        (df_contam.pscore <= max_ps)].copy()
    pct_trimmed = 100 * (1 - len(df_trim) / len(df_contam))

    # (2) Nearest-neighbor match on pscore, no replacement
    pair_diffs = nn_match_pair_diffs(df_trim)

    treated_mean = df_trim[df_trim.treatment==1]['outcome'].mean()
    control_mean = df_trim[df_trim.treatment==0]['outcome'].mean()
    rr_est = treated_mean / control_mean if control_mean > 0 else np.nan
    ev = evalue_ex(rr_est) if rr_est >= 1 else evalue_ex(1/rr_est)

    # (3) Rosenbaum bound
    rb = rosenbaum_bound_ex(pair_diffs)

    return {
        'contamination':   contamination_fraction,
        'n_total':         len(df_contam),
        'pct_trimmed':     round(pct_trimmed, 1),
        'rr_estimate':     round(rr_est, 3),
        'e_value':         round(ev, 2),
        'rosenbaum_bound': rb
    }

print("Exercise 15.1: Break the Model")
fractions = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
ex_results = [run_analysis(f) for f in fractions]
print(pd.DataFrame(ex_results).to_string(index=False))
print()
print("Guided Questions:")
print("Q1 (Mechanical): Track E-value and Rosenbaum Γ as contamination")
print("    increases. Where do they cross benchmarks (e.g. Γ < 1.5, E < 1.8)?")
print("    Does direction match the chapter's 'What You Should Find'?")
print("Q2 (Interpretive): Which statistic degrades faster and why?")
print("Q3 (Evaluative): At 60% contamination, what has gone wrong?")
print("    Identify the failure in the Tetrahedron and write")
print("    the corrected disclosure language.")


Exercise 15.1: Break the Model
 contamination  n_total  pct_trimmed  rr_estimate  e_value  rosenbaum_bound
           0.0     1800          0.1        1.153     1.57              1.7
           0.2     1880          0.1        1.179     1.64              1.7
           0.4     1960          0.1        1.211     1.72              1.8
           0.6     2040          0.1        1.230     1.76              1.9
           0.8     2120          0.1        1.239     1.78              2.0
           1.0     2200          0.1        1.250     1.81              2.0

Guided Questions:
Q1 (Mechanical): Track E-value and Rosenbaum Γ as contamination
    increases. Where do they cross benchmarks (e.g. Γ < 1.5, E < 1.8)?
    Does direction match the chapter's 'What You Should Find'?
Q2 (Interpretive): Which statistic degrades faster and why?
Q3 (Evaluative): At 60% contamination, what has gone wrong?
    Identify the failure in the Tetrahedron and write
    the corrected disclosure language.


In [ ]:
# ============================================================
# CELL 15 — LLM Plain Language E-Value Interpretation
# ============================================================
# ============================================================

import json, urllib.request

def call_claude(prompt):
    url = "https://api.anthropic.com/v1/messages"
    payload = json.dumps({
        "model": "claude-sonnet-4-20250514",
        "max_tokens": 1000,
        "messages": [{"role": "user", "content": prompt}]
    }).encode()
    req = urllib.request.Request(url, data=payload, headers={
        "Content-Type": "application/json",
        "anthropic-version": "2023-06-01"
    })
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())["content"][0]["text"]

# Use verified outputs from Cells 6 and 7
# If API key not set, print a demonstration output instead

PLAIN_LANGUAGE_PROMPT = f"""
A hospital study found that follow-up phone calls after discharge
were associated with reduced 30-day readmission (OR = 0.71).

The sensitivity analysis produced these results:
- E-value (point estimate): 2.17
- E-value (confidence interval bound): 1.56
- Rosenbaum bound: Gamma = 1.6

Write exactly 3 sentences in plain English for a hospital administrator
who is deciding whether to fund a $400,000 expansion of this program.

Rules:
- No statistical jargon (no "odds ratio", no "p-value", no "CI")
- Be honest about fragility — do not oversell the result
- End with a concrete recommendation about what additional evidence
  would be needed before funding the expansion
- Each sentence should be under 40 words
"""

print("LLM Plain Language Interpretation")
print("=" * 50)

try:
    interpretation = call_claude(PLAIN_LANGUAGE_PROMPT)
    print(interpretation)
except Exception:
    # Fallback demonstration if API key not available
    print("[API not available — demonstration output below]")
    print()
    print("The study found that patients who received follow-up calls")
    print("were less likely to be readmitted, but a hidden factor like")
    print("medication adherence — which we did not measure — could")
    print("plausibly explain the entire finding on its own.")
    print()
    print("Our analysis shows the result is moderately fragile: a")
    print("confounder only 60% stronger than chance between matched")
    print("patients would be enough to erase statistical significance.")
    print()
    print("Before committing $400,000, run a prospective pilot where")
    print("call assignment is determined by scheduling logistics rather")
    print("than patient behavior, which would isolate the true effect.")

LLM Plain Language Interpretation
[API not available — demonstration output below]

The study found that patients who received follow-up calls
were less likely to be readmitted, but a hidden factor like
medication adherence — which we did not measure — could
plausibly explain the entire finding on its own.

Our analysis shows the result is moderately fragile: a
confounder only 60% stronger than chance between matched
patients would be enough to erase statistical significance.

Before committing $400,000, run a prospective pilot where
call assignment is determined by scheduling logistics rather
than patient behavior, which would isolate the true effect.


In [2]:
 #============================================================
# CELL 16 — Automated Decision-Ready Sensitivity Report
# ============================================================
# The professor's spec: "An agent that automatically runs a
# sensitivity analysis suite, generates visualizations, and
# produces a decision-ready sensitivity report."
#
# This cell takes ALL outputs from the notebook — AIPW ATE,
# E-values, Rosenbaum bound, positivity check, falsification
# test — and generates a structured one-page report automatically.
#
# The human still makes the final decision. The agent produces
# the briefing document. That is the correct division of labor.
# ============================================================

# Pull verified values from earlier cells
# (these variables must exist from running Cells 5-9)
try:
    ate_val  = round(ate, 4)
    se_val   = round(se, 4)
    ci_lo    = round(ate - 1.96 * se, 4)
    ci_hi    = round(ate + 1.96 * se, 4)
    ev_pt    = round(ev_estimate, 2)
    ev_ci    = round(ev_ci_bound, 2)
    rb_val   = threshold
    n_out    = len(outside)
    pct_out  = round(100 * len(outside) / n, 1)
    p_false  = round(p_val, 4)
except NameError:
    # Fallback to verified values if variables not in scope
    ate_val  = 0.0781
    se_val   = 0.0208
    ci_lo    = 0.0372
    ci_hi    = 0.1189
    ev_pt    = 2.17
    ev_ci    = 1.56
    rb_val   = 1.6
    n_out    = 4
    pct_out  = 0.2
    p_false  = 0.5549

REPORT_PROMPT = f"""
Generate a structured one-page sensitivity report for the following
causal study. Use the exact structure specified below.
Write for a senior hospital administrator — clear, direct, no jargon.

STUDY: Hospital follow-up call program — effect on 30-day readmission

RESULTS:
- AIPW Average Treatment Effect: {ate_val} (95% CI: {ci_lo}, {ci_hi})
- E-value (point estimate): {ev_pt}
- E-value (CI bound): {ev_ci}
- Rosenbaum bound: Gamma = {rb_val}
- Units outside common support: {n_out} ({pct_out}%)
- Falsification test p-value: {p_false}
- Known benchmark: medication adherence has RR ~2.0 in both directions

REQUIRED STRUCTURE — use exactly these five headers:

FINDING (1 sentence):
State what the study found in plain language.

ROBUSTNESS ASSESSMENT (2 sentences):
How strong would a hidden variable need to be to explain away this result?
Is that level of hidden confounding plausible in this clinical context?

KEY THREAT TO VALIDITY (1 sentence):
Name the single most plausible unmeasured confounder and why it matters.

DIAGNOSTIC CHECKS (2 sentences):
What did the positivity check and falsification test find?
What do those results mean for confidence in the estimate?

RECOMMENDATION (2 sentences):
What should the administrator do before funding the expansion?
What specific study design would resolve the remaining uncertainty?
"""

print("\n\nAutomated Decision-Ready Sensitivity Report")
print("=" * 50)

try:
    report = call_claude(REPORT_PROMPT)
    print(report)
except Exception:
    print("[API not available — demonstration output below]")
    print()
    print("FINDING:")
    print("Patients who received a follow-up call after discharge were")
    print("approximately 7.8 percentage points more likely to avoid")
    print("readmission within 30 days, after adjusting for age, severity,")
    print("insurance status, and length of stay.")
    print()
    print("ROBUSTNESS ASSESSMENT:")
    print("An unmeasured variable would need to be associated with both")
    print("call receipt and readmission avoidance by a factor of 2.17 to")
    print("fully explain away the finding. Medication adherence — which")
    print("was not measured — has associations of roughly this magnitude,")
    print("making the result moderately fragile.")
    print()
    print("KEY THREAT TO VALIDITY:")
    print("Patient motivation and health literacy predict both who answers")
    print("calls and who avoids readmission, and neither was measured.")
    print()
    print("DIAGNOSTIC CHECKS:")
    print("The positivity check found 4 patients (0.2%) outside common")
    print("support — predominantly weekend discharges — who were trimmed")
    print("before analysis. The falsification test passed (p = 0.5549),")
    print("meaning the treatment variable does not predict pre-discharge")
    print("outcomes it causally cannot affect.")
    print()
    print("RECOMMENDATION:")
    print("Do not approve the $400,000 expansion on this evidence alone.")
    print("Commission a prospective pilot where call assignment is")
    print("determined by scheduling logistics rather than patient behavior,")
    print("which would isolate the causal effect from selection bias.")



Automated Decision-Ready Sensitivity Report
[API not available — demonstration output below]

FINDING:
Patients who received a follow-up call after discharge were
approximately 7.8 percentage points more likely to avoid
readmission within 30 days, after adjusting for age, severity,
insurance status, and length of stay.

ROBUSTNESS ASSESSMENT:
An unmeasured variable would need to be associated with both
call receipt and readmission avoidance by a factor of 2.17 to
fully explain away the finding. Medication adherence — which
was not measured — has associations of roughly this magnitude,
making the result moderately fragile.

KEY THREAT TO VALIDITY:
Patient motivation and health literacy predict both who answers
calls and who avoids readmission, and neither was measured.

DIAGNOSTIC CHECKS:
The positivity check found 4 patients (0.2%) outside common
support — predominantly weekend discharges — who were trimmed
before analysis. The falsification test passed (p = 0.5549),
meaning the trea